<a href="https://colab.research.google.com/github/vkjadon/hugging_face/blob/main/attention_computation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Computing Transformer Internals Using Real Models and Libraries

This tutorial demonstrates how to compute:

1. Tokenization
2. Word embeddings using Word2Vec
3. Positional encoding
4. Transformer encoder self-attention
5. Decoder masked attention
6. Cross attention
7. Final probability prediction

for the sentence:

```text
The drone is flying
```

using real Python libraries and modern NLP tools.


# 1. Install Required Libraries

In [ ]:
!pip install gensim

# 2. Import Libraries

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModel
from gensim.models import Word2Vec

# 3. Input Sentence

In [ ]:
sentence = "The drone is flying"

# 4. Tokenization Using Hugging Face Tokenizer

In [ ]:
from transformers import BertTokenizer

# Load tokenizer
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize
encoded = bert_tokenizer(sentence, return_tensors='pt')

print("Input IDs:")
print(encoded['input_ids'])

print("Tokens:")
print(bert_tokenizer.convert_ids_to_tokens(encoded['input_ids'][0]))

We use the BERT tokenizer.

Explanation:

| Token | Meaning     |
| ----- | ----------- |
| [CLS] | Start token |
| [SEP] | End token   |

---


# 5. Word Embeddings Using Word2Vec

Now we generate embeddings using Word2Vec.

---

## 5.1 Prepare Training Corpus


In [ ]:
corpus = [
    ["the", "drone", "is", "flying"],
    ["the", "airplane", "is", "landing"],
    ["a", "bird", "is", "flying"],
    ["the", "helicopter", "is", "hovering"],
    ["the", "pilot", "controls", "the", "drone"]
]

## 5.2 Train Word2Vec Model


In [ ]:
w2v_model = Word2Vec(
    sentences=corpus,
    vector_size=8,
    window=2,
    min_count=1,
    workers=1
)

## 5.3 Extract Embeddings

In [ ]:
words = ["the", "drone", "is", "flying"]

embeddings = []

for word in words:
    vector = w2v_model.wv[word]
    embeddings.append(vector)

embeddings = np.array(embeddings)

print("Embedding Shape:")
print(embeddings.shape)

print("Embeddings:")
print(embeddings)



Output Example:

```python
(4, 8)
```

Meaning:

* 4 tokens
* 8-dimensional embeddings

---


## 7.2 Compute Positional Encodings

In [ ]:
import math

def positional_encoding(seq_len, d_model):

    PE = np.zeros((seq_len, d_model))

    for pos in range(seq_len):

        for i in range(0, d_model, 2):

            denominator = np.power(10000, (2 * i) / d_model)

            PE[pos, i] = np.sin(pos / denominator)

            if i + 1 < d_model:
                PE[pos, i + 1] = np.cos(pos / denominator)

    return PE

In [ ]:
PE = positional_encoding(seq_len=4, d_model=8)

print(PE)

# 8. Add Embeddings + Positional Encoding

In [ ]:
encoder_input = embeddings + PE

print(encoder_input)

# 9. Convert to PyTorch Tensor

In [ ]:
encoder_input = torch.tensor(encoder_input, dtype=torch.float32)

print(encoder_input.shape)

This becomes the input to the Transformer encoder.


# 10. Computing Query, Key, and Value

Transformers create:

* Query (Q)
* Key (K)
* Value (V)

using learnable matrices.

---

In [ ]:
## 10.1 Create Linear Layers

embedding_dim = 8

Wq = nn.Linear(embedding_dim, embedding_dim)
Wk = nn.Linear(embedding_dim, embedding_dim)
Wv = nn.Linear(embedding_dim, embedding_dim)

In [ ]:
## 10.2 Generate Q, K, V

Q = Wq(encoder_input)
K = Wk(encoder_input)
V = Wv(encoder_input)

print("Q Shape:", Q.shape)
print("K Shape:", K.shape)
print("V Shape:", V.shape)


# 11. Self Attention Computation

Attention Formula:

```text
Attention(Q,K,V) = softmax(QK^T / sqrt(dk)) V
```

---

## 11.1 Compute Attention Scores

```python
dk = K.shape[-1]

scores = torch.matmul(Q, K.T) / math.sqrt(dk)

print(scores)
```

---

## 11.2 Apply Softmax

```python
attention_weights = torch.softmax(scores, dim=-1)

print(attention_weights)
```

Each row shows how much a token attends to other tokens.

---

## 11.3 Compute Attention Output

```python
attention_output = torch.matmul(attention_weights, V)

print(attention_output)
```

Now embeddings become context-aware.

---

# 12. Multi-Head Attention Using PyTorch

Instead of manually computing attention, PyTorch provides:

```python
nn.MultiheadAttention
```

---

## 12.1 Create Multihead Attention Layer

```python
multihead = nn.MultiheadAttention(
    embed_dim=8,
    num_heads=2,
    batch_first=True
)
```

---

## 12.2 Prepare Input

PyTorch expects:

```text
(batch_size, sequence_length, embedding_dimension)
```

```python
encoder_batch = encoder_input.unsqueeze(0)

print(encoder_batch.shape)
```

Output:

```python
torch.Size([1, 4, 8])
```

---

## 12.3 Run Multihead Attention

```python
output, weights = multihead(
    encoder_batch,
    encoder_batch,
    encoder_batch
)

print("Output Shape:")
print(output.shape)

print("Attention Weights:")
print(weights)
```

---

# 13. Feed Forward Network

Each Transformer block contains a feed-forward network.

---

## 13.1 Define FFN

```python
ffn = nn.Sequential(
    nn.Linear(8, 16),
    nn.ReLU(),
    nn.Linear(16, 8)
)
```

---

## 13.2 Compute FFN Output

```python
ffn_output = ffn(output)

print(ffn_output)
```

---

# 14. Residual Connection + Layer Normalization

---

## 14.1 Residual Connection

```python
residual_output = output + ffn_output
```

---

## 14.2 Layer Normalization

```python
layer_norm = nn.LayerNorm(8)

normalized_output = layer_norm(residual_output)

print(normalized_output)
```

This completes one encoder block.

---

# 15. Decoder Masked Self Attention

Decoder cannot see future words.

Suppose decoder input:

```text
<START> The drone
```

---

## 15.1 Create Decoder Embeddings

```python
decoder_tokens = torch.rand((1, 4, 8))
```

---

## 15.2 Create Causal Mask

```python
mask = torch.triu(torch.ones(4, 4) * float('-inf'), diagonal=1)

print(mask)
```

---

## 15.3 Apply Masked Attention

```python
masked_output, masked_weights = multihead(
    decoder_tokens,
    decoder_tokens,
    decoder_tokens,
    attn_mask=mask
)

print(masked_weights)
```

Future positions now become invisible.

---

# 16. Cross Attention

Decoder attends to encoder outputs.

---

## 16.1 Cross Attention Computation

```python
cross_output, cross_weights = multihead(
    decoder_tokens,
    output,
    output
)

print(cross_output)
```

Meaning:

| Component | Source  |
| --------- | ------- |
| Query     | Decoder |
| Key       | Encoder |
| Value     | Encoder |

---

# 17. Final Vocabulary Prediction

Transformer predicts the next word.

Suppose vocabulary size = 20.

---

## 17.1 Final Linear Layer

```python
vocab_size = 20

final_layer = nn.Linear(8, vocab_size)

logits = final_layer(cross_output)

print(logits.shape)
```

Output:

```python
torch.Size([1, 4, 20])
```

---

## 17.2 Convert to Probabilities

```python
probabilities = torch.softmax(logits, dim=-1)

print(probabilities)
```

Each token now has probabilities over the vocabulary.

---

# 18. Predict Next Token

```python
predicted_token = torch.argmax(probabilities, dim=-1)

print(predicted_token)
```

Example Output:

```python
tensor([[5, 2, 9, 11]])
```

These IDs map back to vocabulary words.

---

# 19. Using Real BERT Embeddings

Instead of Word2Vec, we can use pretrained BERT embeddings.

---

## 19.1 Load BERT Model

```python
from transformers import BertModel

bert_model = BertModel.from_pretrained('bert-base-uncased')
```

---

## 19.2 Get Embeddings

```python
with torch.no_grad():
    outputs = bert_model(**encoded)

last_hidden_state = outputs.last_hidden_state

print(last_hidden_state.shape)
```

Output:

```python
torch.Size([1, 6, 768])
```

Meaning:

| Dimension | Meaning             |
| --------- | ------------------- |
| 1         | Batch size          |
| 6         | Number of tokens    |
| 768       | Embedding dimension |

---

# 20. Extract Attention Weights from BERT

---

## 20.1 Enable Attention Outputs

```python
bert_model = BertModel.from_pretrained(
    'bert-base-uncased',
    output_attentions=True
)
```

---

## 20.2 Run Model

```python
with torch.no_grad():
    outputs = bert_model(**encoded)
```

---

## 20.3 Get Attention Weights

```python
attentions = outputs.attentions

print(len(attentions))
```

Output:

```python
12
```

Meaning:

* 12 Transformer layers

---

## 20.4 Visualize Attention Matrix

```python
attention_matrix = attentions[0][0][0]

plt.imshow(attention_matrix.numpy())
plt.colorbar()
plt.title("Attention Matrix")
plt.show()
```

This shows how tokens attend to each other.

---

# 21. Complete Transformer Pipeline

```text
Sentence
   ↓
Tokenizer
   ↓
Token IDs
   ↓
Embeddings
   ↓
Positional Encoding
   ↓
Query-Key-Value
   ↓
Self Attention
   ↓
Multihead Attention
   ↓
Feed Forward Network
   ↓
Residual + LayerNorm
   ↓
Decoder Masked Attention
   ↓
Cross Attention
   ↓
Linear Layer
   ↓
Softmax
   ↓
Predicted Word
```

---

# 22. Important Real-World Notes

Modern LLMs use:

| Component           | Typical Size          |
| ------------------- | --------------------- |
| Vocabulary          | 50k–250k tokens       |
| Embedding dimension | 768–12288             |
| Layers              | 12–120                |
| Attention heads     | 12–128                |
| Parameters          | Millions to trillions |

---

# 23. Recommended Experiments

You can now try:

1. Changing the sentence
2. Increasing embedding dimensions
3. Increasing attention heads
4. Visualizing attention maps
5. Using GPT-2 tokenizer
6. Comparing Word2Vec vs BERT embeddings
7. Building your own mini-transformer

---

# 24. Summary

In this tutorial, we computed Transformer internals using:

| Component           | Library         |
| ------------------- | --------------- |
| Tokenization        | Hugging Face    |
| Embeddings          | Word2Vec / BERT |
| Attention           | PyTorch         |
| Multihead Attention | PyTorch         |
| Visualization       | Matplotlib      |

This provides a practical understanding of how modern Transformer models process language internally.
